## Notebook 03 — Bronze to Silver (Cleanse & Standardize)

Read Bronze, apply validation rules (DQX), parse/standardize violations, save to Silver.

**Validation Rules:**
1. Restaurant Name, Inspection Date & Type, Zip cannot be null
2. Zip must be valid format
3. Dallas score cannot exceed 100
4. Chicago Results cannot be null
5. Every inspection ≥ 1 unique violation; duplicates loaded as distinct
6. Dallas: score ≥ 90 → max 3 violations
7. Dallas: no PASS if Urgent/Critical violation
8. Chicago: derive violation score from Results


In [0]:
%pip install databricks-labs-dqx

In [0]:
%restart_python

In [0]:
CATALOG = "chicago_dallas_food_inspection"

---
## Part A — Chicago: Bronze → Silver

### 3A.1 — Load Bronze & Type Cast
Columns already renamed in Bronze. Apply type casting, drop metadata columns, trim strings.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

df_chi = spark.table(f"{CATALOG}.bronze.chicago_raw")

# Columns already renamed in Bronze — just type cast and cleanup
df_chi = df_chi.withColumn("Inspection_Date", to_date(col("Inspection_Date"), "MM/dd/yyyy"))
df_chi = df_chi.withColumn("License_Num", col("License_Num").cast("string"))
df_chi = df_chi.withColumn("Latitude", col("Latitude").cast("double"))
df_chi = df_chi.withColumn("Longitude", col("Longitude").cast("double"))
df_chi = df_chi.drop("Location", "_ingestion_timestamp", "_source_city")

for f in df_chi.schema.fields:
    if isinstance(f.dataType, StringType):
        df_chi = df_chi.withColumn(f.name, trim(col(f.name)))

print(f"Chicago cleaned: {df_chi.count()} rows")

### 3A.2 — Parse Violations & Derive Score

In [0]:
# Split pipe-delimited violations
df_chi = df_chi.withColumn("Violations_Array",
    when(col("Violations").isNotNull() & (col("Violations") != ""), split(col("Violations"), "\\|")))

# Deduplicate within each inspection
df_chi = df_chi.withColumn("Violations_Distinct", 
    when(col("Violations_Array").isNotNull(), array_distinct(col("Violations_Array"))))

# Count unique violations
df_chi = df_chi.withColumn("Violation_Count",
    when(col("Violations_Distinct").isNotNull(), size(col("Violations_Distinct"))).otherwise(lit(0)))

# Derive score from Results
df_chi = df_chi.withColumn("Violation_Score",
    when(col("Results") == "Pass", 90)
    .when(col("Results") == "Pass w/ Conditions", 80)
    .when(col("Results") == "Fail", 70)
    .when(col("Results") == "No Entry", 0)
    .otherwise(lit(None).cast("int")))

display(df_chi.groupBy("Results", "Violation_Score").count().orderBy("Violation_Score"))

### 3A.3 — Explode Violations (Standardized Schema)

In [0]:
# Explode each violation into its own row
df_chi_viol = (
    df_chi.filter(col("Violations_Distinct").isNotNull())
    .select("Inspection_ID", "DBA_Name", "AKA_Name", "License_Num",
            "Facility_Type", "Risk", "Address", "City", "State", "Zip",
            "Inspection_Date", "Inspection_Type", "Results", "Violation_Score",
            "Latitude", "Longitude",
            explode(col("Violations_Distinct")).alias("Violation_Raw"))
    .withColumn("Violation_Raw", trim(col("Violation_Raw")))
)

# Parse CODE. DESCRIPTION - Comments: TEXT
df_chi_viol = (df_chi_viol
    .withColumn("Violation_Code",
    regexp_extract(col("Violation_Raw"), r"^(\d+)\.", 1))
.withColumn("Violation_Code",
    when(col("Violation_Code") != "", col("Violation_Code").cast("int")))
    .withColumn("Violation_Description",
        when(col("Violation_Raw").contains("- Comments:"),
             trim(regexp_extract(col("Violation_Raw"), r"^\d+\.\s*(.+?)\s*-\s*Comments:", 1)))
        .otherwise(trim(regexp_extract(col("Violation_Raw"), r"^\d+\.\s*(.+)$", 1))))
    .withColumn("Violation_Comments",
        when(col("Violation_Raw").contains("- Comments:"),
             trim(regexp_extract(col("Violation_Raw"), r"-\s*Comments:\s*(.*)", 1))))
    .withColumn("Source_City", lit("Chicago"))
    .drop("Violation_Raw"))

print(f"Chicago violations (exploded): {df_chi_viol.count()}")
display(df_chi_viol.select("Inspection_ID", "Violation_Code", "Violation_Description").limit(10))

### 3A.4 — DQX Validation

In [0]:
import yaml
from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()
dqengine = DQEngine(ws)

df_chi_qc = df_chi.withColumn("dup_count", count("*").over(Window.partitionBy("Inspection_ID")))

chi_checks = yaml.safe_load("""
- criticality: error
  check: {function: sql_expression, arguments: {expression: "DBA_Name IS NOT NULL AND TRIM(DBA_Name) != ''", msg: "Restaurant Name null"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "Inspection_Date IS NOT NULL", msg: "Inspection_Date null"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "Inspection_Type IS NOT NULL AND TRIM(Inspection_Type) != ''", msg: "Inspection_Type null"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "Results IS NOT NULL AND TRIM(Results) != ''", msg: "Chicago Results null"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "Zip IS NOT NULL AND TRIM(Zip) != ''", msg: "Zip null"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "Zip IS NULL OR TRIM(Zip) = '' OR Zip RLIKE '^[0-9]{5}(-[0-9]{4})?$'", msg: "Invalid Zip format"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "Violation_Count >= 1", msg: "Must have >= 1 violation"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "Inspection_Date <= current_date()", msg: "Future date"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "dup_count = 1", msg: "Duplicate Inspection_ID"}}
""")

chi_valid, chi_quarantine = dqengine.apply_checks_by_metadata_and_split(df_chi_qc, chi_checks, globals())
print(f"Chicago — Total: {df_chi_qc.count()}, Valid: {chi_valid.count()}, Quarantine: {chi_quarantine.count()}")

In [0]:
from pyspark.sql import functions as F
display(chi_quarantine.select(F.explode("_errors").alias("e")).groupBy(F.col("e.name").alias("rule")).count().orderBy(F.desc("count")))

### 3A.5 — Save Chicago Silver

In [0]:
chi_drop = ["_errors", "_warnings", "dup_count", "Violations", "Violations_Array", "Violations_Distinct", "Violation_Count"]
df_chi_silver = chi_valid.select(*[c for c in chi_valid.columns if c not in chi_drop])

# Save to Silver (NULLs preserved — sentinel values applied only in Gold per Kimball)
df_chi_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.chicago_inspections")

valid_ids = df_chi_silver.select("Inspection_ID")
df_chi_viol_silver = df_chi_viol.join(valid_ids, "Inspection_ID", "inner")
df_chi_viol_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.chicago_violations")

chi_quarantine.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.chicago_quarantine")

print(f"Chicago Silver — Inspections: {df_chi_silver.count()}, Violations: {df_chi_viol_silver.count()}, Quarantine: {chi_quarantine.count()}")

---
## Part B — Dallas: Bronze → Silver

### 3B.1 — Load Bronze & Type Cast
Columns already renamed in Bronze. Apply type casting, parse Lat/Long, trim strings.

In [0]:
df_dal = spark.table(f"{CATALOG}.bronze.dallas_raw")

# Columns already renamed in Bronze — just type cast and cleanup
df_dal = df_dal.withColumn("Inspection_Date", to_date(col("Inspection_Date"), "MM/dd/yyyy"))
df_dal = df_dal.withColumn("Inspection_Score", col("Inspection_Score").cast("int"))

# Parse Lat Long into separate double columns
# Parse Lat Long into separate double columns (use try_cast for empty strings)
df_dal = (df_dal
    .withColumn("Latitude", expr("try_cast(regexp_extract(Lat_Long_Location, '\\\\(([\\\\-]?[\\\\d.]+),\\\\s*([\\\\-]?[\\\\d.]+)\\\\)', 1) AS DOUBLE)"))
    .withColumn("Longitude", expr("try_cast(regexp_extract(Lat_Long_Location, '\\\\(([\\\\-]?[\\\\d.]+),\\\\s*([\\\\-]?[\\\\d.]+)\\\\)', 2) AS DOUBLE)"))
)

# Cast Violation_Points to int
for v in range(1, 26):
    df_dal = df_dal.withColumn(f"Violation_Points_{v}", col(f"Violation_Points_{v}").cast("int"))

df_dal = df_dal.drop("_ingestion_timestamp", "_source_city")

for f in df_dal.schema.fields:
    if isinstance(f.dataType, StringType):
        df_dal = df_dal.withColumn(f.name, trim(col(f.name)))

print(f"Dallas cleaned: {df_dal.count()} rows")

### 3B.2 — Violation Count & Urgent/Critical Flag

In [0]:
from functools import reduce
from operator import add

viol_desc_cols = [f"Violation_Description_{v}" for v in range(1, 26)]
df_dal = df_dal.withColumn("Violation_Count",
    reduce(add, [when(col(c).isNotNull() & (col(c) != ""), lit(1)).otherwise(lit(0)) for c in viol_desc_cols]))

urgent_parts = [f"UPPER(COALESCE(Violation_Description_{v}, '')) LIKE '%URGENT%' OR UPPER(COALESCE(Violation_Description_{v}, '')) LIKE '%CRITICAL%'" for v in range(1, 26)]
df_dal = df_dal.withColumn("Has_Urgent_Critical", expr(f"CASE WHEN ({' OR '.join(urgent_parts)}) THEN true ELSE false END"))

print(f"Score >= 90 + > 3 violations: {df_dal.filter('Inspection_Score >= 90 AND Violation_Count > 3').count()}")
print(f"Score >= 90 + Urgent/Critical: {df_dal.filter('Has_Urgent_Critical AND Inspection_Score >= 90').count()}")

### 3B.3 — Explode Violations (Standardized Schema)

In [0]:
from pyspark.sql.functions import regexp_replace, regexp_extract, trim, when, col, struct, array, explode, lit

violation_structs = []
for v in range(1, 26):
    violation_structs.append(
        when(col(f"Violation_Description_{v}").isNotNull() & (col(f"Violation_Description_{v}") != ""),
             struct(col(f"Violation_Description_{v}").alias("Violation_Description"),
                    col(f"Violation_Points_{v}").alias("Violation_Points"),
                    col(f"Violation_Detail_{v}").alias("Violation_Detail"),
                    col(f"Violation_Memo_{v}").alias("Violation_Memo"))))

df_dal = df_dal.withColumn("Violations_Struct_Array", array(*violation_structs))

df_dal_viol = (
    df_dal.select("Restaurant_Name", "Inspection_Type", "Inspection_Date", "Inspection_Score",
                  "Street_Address", "Zip_Code", "Latitude", "Longitude",
                  explode(col("Violations_Struct_Array")).alias("v"))
    .filter(col("v").isNotNull())
    .select("Restaurant_Name", "Inspection_Date", "Inspection_Score", "Street_Address", "Zip_Code",
            col("v.Violation_Description").alias("Violation_Description"),
            col("v.Violation_Points").alias("Violation_Points"),
            col("v.Violation_Detail").alias("Violation_Detail"),
            col("v.Violation_Memo").alias("Violation_Comments"),
            lit("Dallas").alias("Source_City"))
)

# Step 1: Extract violation code — handle "* 21" format (space after asterisk)
df_dal_viol = df_dal_viol.withColumn("Violation_Code",
    regexp_extract(col("Violation_Description"), r"^\*?\s*(\d+)", 1))
df_dal_viol = df_dal_viol.withColumn("Violation_Code",
    when(col("Violation_Code") != "", col("Violation_Code").cast("int")))

# Step 2: Clean description — strip "*CODE " or "* CODE " prefix
df_dal_viol = df_dal_viol.withColumn("Violation_Description",
    trim(regexp_replace(col("Violation_Description"), r"^\*?\s*\d+\s*", "")))

# Step 2b: Strip lone asterisk with no digit (e.g., "* CERTIFIED FOOD MANAGER...")
df_dal_viol = df_dal_viol.withColumn("Violation_Description",
    trim(regexp_replace(col("Violation_Description"), r"^\*\s*", "")))

# Deduplicate violations within each inspection
df_dal_viol = df_dal_viol.dropDuplicates(["Restaurant_Name", "Inspection_Date", "Violation_Description"])

print(f"Dallas violations (exploded, cleaned & deduped): {df_dal_viol.count()}")
display(df_dal_viol.select("Restaurant_Name", "Violation_Code", "Violation_Description", "Violation_Points").limit(10))

### 3B.4 — DQX Validation

In [0]:
dal_key = ["Restaurant_Name", "Inspection_Date", "Inspection_Type", "Zip_Code"]
df_dal_qc = df_dal.withColumn("dup_count", count("*").over(Window.partitionBy(*dal_key)))

dal_checks = yaml.safe_load("""
- criticality: error
  check: {function: sql_expression, arguments: {expression: "Restaurant_Name IS NOT NULL AND TRIM(Restaurant_Name) != ''", msg: "Restaurant_Name null"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "Inspection_Date IS NOT NULL", msg: "Inspection_Date null"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "Inspection_Type IS NOT NULL AND TRIM(Inspection_Type) != ''", msg: "Inspection_Type null"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "Zip_Code IS NOT NULL AND TRIM(Zip_Code) != ''", msg: "Zip_Code null"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "Zip_Code IS NULL OR TRIM(Zip_Code) = '' OR Zip_Code RLIKE '^[0-9]{5}(-[0-9]{4})?$'", msg: "Invalid Zip format"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "Inspection_Score <= 100", msg: "Score > 100"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "Inspection_Score >= 0", msg: "Negative score"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "NOT (Inspection_Score >= 90 AND Violation_Count > 3)", msg: "Score>=90 with >3 violations"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "NOT (Inspection_Score >= 90 AND Has_Urgent_Critical = true)", msg: "PASS with Urgent/Critical"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "Violation_Count >= 1", msg: "Must have >= 1 violation"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "Inspection_Date <= current_date()", msg: "Future date"}}
- criticality: error
  check: {function: sql_expression, arguments: {expression: "dup_count = 1", msg: "Duplicate"}}
""")

dal_valid, dal_quarantine = dqengine.apply_checks_by_metadata_and_split(df_dal_qc, dal_checks, globals())
print(f"Dallas — Total: {df_dal_qc.count()}, Valid: {dal_valid.count()}, Quarantine: {dal_quarantine.count()}")

In [0]:
display(dal_quarantine.select(F.explode("_errors").alias("e")).groupBy(F.col("e.name").alias("rule")).count().orderBy(F.desc("count")))

### 3B.5 — Save Dallas Silver

In [0]:
# Drop wide violation cols + internal cols
dal_drop = ["_errors", "_warnings", "dup_count", "Violations_Struct_Array", "Lat_Long_Location", "Has_Urgent_Critical", "Violation_Count"]
for v in range(1, 26):
    dal_drop.extend([f"Violation_Description_{v}", f"Violation_Points_{v}", f"Violation_Detail_{v}", f"Violation_Memo_{v}"])

df_dal_silver = dal_valid.select(*[c for c in dal_valid.columns if c not in dal_drop])

# Save to Silver (NULLs preserved — sentinel values applied only in Gold per Kimball)
df_dal_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.dallas_inspections")

dal_keys = df_dal_silver.select("Restaurant_Name", "Inspection_Date").distinct()
df_dal_viol_silver = df_dal_viol.join(dal_keys, ["Restaurant_Name", "Inspection_Date"], "inner")
df_dal_viol_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.dallas_violations")

dal_quarantine.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.dallas_quarantine")

print(f"Dallas Silver — Inspections: {df_dal_silver.count()}, Violations: {df_dal_viol_silver.count()}, Quarantine: {dal_quarantine.count()}")

---
### Silver Zone Summary

In [0]:
print("=" * 70); print("SILVER ZONE SUMMARY"); print("=" * 70)
for t in ["chicago_inspections", "chicago_violations", "chicago_quarantine",
          "dallas_inspections", "dallas_violations", "dallas_quarantine"]:
    print(f"  {CATALOG}.silver.{t}: {spark.table(f'{CATALOG}.silver.{t}').count()} rows")
print("=" * 70)